## Setup
Importing `pandas`/`numpy` for data handling, `scikit-learn` for model training and splitting, and metric functions to score each model the same way.

In [ ]:
# Step 1: Import Necessary Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report


## Load the Data
Using Colab's upload widget to bring in `SmartWatchData.csv` — 120,000 customer records, 40 variables — into the session.

In [ ]:
# Step 2: Load Data
from google.colab import files
uploaded = files.upload()

Saving SmartWatchData.csv to SmartWatchData.csv


Reading the CSV into a DataFrame and confirming its shape (120,000 rows × 40 columns, no missing values).

In [ ]:
import io
import pandas as pd
df = pd.read_csv('SmartWatchData.csv')
print('Data shape:', df.shape)

Data shape: (120000, 40)


## Define Features (X) and Target (y)
`Adopt` is the outcome we're predicting (1 = purchased the smartwatch, 0 = did not). Every other column becomes a predictor.

In [ ]:
# Step 3: Define Features and Target Variable
# Our target variable 'Adopt' indicates whether the individual bought a smartwatch.
X = df.drop(columns=['Adopt'])
y = df['Adopt']

## Train/Test Split
Splitting 70/30 into training (84,000 rows) and test (36,000 rows) sets, so models are evaluated on data they never saw during fitting. `random_state=42` keeps the split reproducible.

In [ ]:
# Step 4: Split the Data into Training and Testing Sets
# We use a 70/30 split. The training set is used to build our models, and the test set evaluates their performance.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print("Train-test split completed.")
print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

Train-test split completed.
Training set shape: (84000, 39)
Test set shape: (36000, 39)


## Evaluation Helper
A small function that prints Accuracy, Precision, Recall, F1, and ROC AUC so every model below is scored the same way.

In [ ]:
# Step 5: Define a Function to Print Evaluation Metrics
def print_classification_metrics(model_name, y_true, y_pred, y_proba):
    print(f"--- {model_name} ---")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))
    print("F1 Score:", f1_score(y_true, y_pred))
    print("ROC AUC:", roc_auc_score(y_true, y_proba))
    print("\nClassification Report:\n", classification_report(y_true, y_pred))
    print("\n")

## Model 1: Logistic Regression (No Penalty)
Baseline linear model, no regularization — every variable stays in, even weak or noisy ones. Expect this to be the weakest performer since it can't capture non-linear patterns.

In [ ]:
# Step 6: Train and Evaluate Classification Models

# 6.1 Logistic Regression without Regularization (Baseline)
# Note: Use penalty=None to indicate no regularization.
lr = LogisticRegression(penalty=None, solver='lbfgs', max_iter=1000)
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)
pred_lr_proba = lr.predict_proba(X_test)[:, 1]
print_classification_metrics("Logistic Regression (No Penalty)", y_test, pred_lr, pred_lr_proba)


--- Logistic Regression (No Penalty) ---
Accuracy: 0.7290555555555556
Precision: 0.7272727272727273
Recall: 0.6770358980424626
F1 Score: 0.7012557427258805
ROC AUC: 0.8097011911837759

Classification Report:
               precision    recall  f1-score   support

           0       0.73      0.78      0.75     19091
           1       0.73      0.68      0.70     16909

    accuracy                           0.73     36000
   macro avg       0.73      0.73      0.73     36000
weighted avg       0.73      0.73      0.73     36000





/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Model 2: Ridge (L2) Logistic Regression
Adds an L2 penalty that shrinks coefficients toward zero to reduce overfitting. It keeps every variable in the model — no automatic feature dropping.

In [ ]:
# 6.2 Logistic Regression with Ridge (L2) Regularization
lr_ridge = LogisticRegression(penalty='l2', solver='lbfgs', max_iter=1000)
lr_ridge.fit(X_train, y_train)
pred_lr_ridge = lr_ridge.predict(X_test)
pred_lr_ridge_proba = lr_ridge.predict_proba(X_test)[:, 1]
print_classification_metrics("Logistic Regression (Ridge - L2)", y_test, pred_lr_ridge, pred_lr_ridge_proba)

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


--- Logistic Regression (Ridge - L2) ---
Accuracy: 0.7298055555555556
Precision: 0.7241293221819998
Recall: 0.6861434738896446
F1 Score: 0.7046248215966718
ROC AUC: 0.8092881862705008

Classification Report:
               precision    recall  f1-score   support

           0       0.73      0.77      0.75     19091
           1       0.72      0.69      0.70     16909

    accuracy                           0.73     36000
   macro avg       0.73      0.73      0.73     36000
weighted avg       0.73      0.73      0.73     36000





## Model 3: Lasso (L1) Logistic Regression
Adds an L1 penalty, which in theory can shrink some coefficients to exactly zero for automatic feature selection. **Caveat:** in this run, none of the 39 coefficients actually hit zero — likely because the features weren't standardized first, which matters a lot for L1 penalties. Worth knowing before calling this "automatic feature selection."

In [ ]:
# 6.3 Logistic Regression with Lasso (L1) Regularization
lr_lasso = LogisticRegression(penalty='l1', solver='saga', max_iter=1000)
lr_lasso.fit(X_train, y_train)
pred_lr_lasso = lr_lasso.predict(X_test)
pred_lr_lasso_proba = lr_lasso.predict_proba(X_test)[:, 1]
print_classification_metrics("Logistic Regression (Lasso - L1)", y_test, pred_lr_lasso, pred_lr_lasso_proba)

--- Logistic Regression (Lasso - L1) ---
Accuracy: 0.7291666666666666
Precision: 0.7179830704585591
Recall: 0.697261813235555
F1 Score: 0.7074707470747075
ROC AUC: 0.8066230775412311

Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.76      0.75     19091
           1       0.72      0.70      0.71     16909

    accuracy                           0.73     36000
   macro avg       0.73      0.73      0.73     36000
weighted avg       0.73      0.73      0.73     36000





/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


## Model 4: Random Forest Classifier
An ensemble of 100 decision trees. This is the model expected to perform best, since it can capture non-linear interactions the three linear models above can't.

In [ ]:
# 6.4 Random Forest Classifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
pred_rf_proba = rf.predict_proba(X_test)[:, 1]
print_classification_metrics("Random Forest Classifier", y_test, pred_rf, pred_rf_proba)

--- Random Forest Classifier ---
Accuracy: 0.8416944444444444
Precision: 0.8250782971813014
Recall: 0.8413271039091608
F1 Score: 0.8331234810108049
ROC AUC: 0.926471801798508

Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.84      0.85     19091
           1       0.83      0.84      0.83     16909

    accuracy                           0.84     36000
   macro avg       0.84      0.84      0.84     36000
weighted avg       0.84      0.84      0.84     36000





## Comparing All Models
Collecting every model's metrics into one table for a side-by-side comparison. Random Forest comes out clearly ahead (~84% accuracy vs. ~73% for the linear models).

In [ ]:
# Combine all model results into one table
# === Collecting Performance Metrics into a Single Table ===
import pandas as pd

# Create a list to hold the metrics dictionaries for each model
metrics_list = []

# Logistic Regression (No Penalty)
metrics_list.append({
    'Model': 'Logistic Regression (No Penalty)',
    'Accuracy': accuracy_score(y_test, pred_lr),
    'Precision': precision_score(y_test, pred_lr),
    'Recall': recall_score(y_test, pred_lr),
    'F1 Score': f1_score(y_test, pred_lr),
    'ROC AUC': roc_auc_score(y_test, pred_lr_proba)
})

# Logistic Regression with Ridge (L2)
metrics_list.append({
    'Model': 'Logistic Regression (Ridge - L2)',
    'Accuracy': accuracy_score(y_test, pred_lr_ridge),
    'Precision': precision_score(y_test, pred_lr_ridge),
    'Recall': recall_score(y_test, pred_lr_ridge),
    'F1 Score': f1_score(y_test, pred_lr_ridge),
    'ROC AUC': roc_auc_score(y_test, pred_lr_ridge_proba)
})

# Logistic Regression with Lasso (L1)
metrics_list.append({
    'Model': 'Logistic Regression (Lasso - L1)',
    'Accuracy': accuracy_score(y_test, pred_lr_lasso),
    'Precision': precision_score(y_test, pred_lr_lasso),
    'Recall': recall_score(y_test, pred_lr_lasso),
    'F1 Score': f1_score(y_test, pred_lr_lasso),
    'ROC AUC': roc_auc_score(y_test, pred_lr_lasso_proba)
})

# Random Forest Classifier
metrics_list.append({
    'Model': 'Random Forest Classifier',
    'Accuracy': accuracy_score(y_test, pred_rf),
    'Precision': precision_score(y_test, pred_rf),
    'Recall': recall_score(y_test, pred_rf),
    'F1 Score': f1_score(y_test, pred_rf),
    'ROC AUC': roc_auc_score(y_test, pred_rf_proba)
})


# Create a DataFrame from the list of metrics
metrics_df = pd.DataFrame(metrics_list)

# Display the table
print(metrics_df)

## Takeaways
Random Forest beats the linear baselines by ~11 points of accuracy (84% vs. ~73%), confirming non-linear modeling adds real value here. Two honest caveats for discussion: Lasso didn't actually zero out any coefficients in this run, and no hyperparameter tuning or feature-importance analysis has been added yet — both are natural next steps.